# Flood Detection and Segmentation using Prithvi-EO-2.0
*AISC Hackathon 2026 — Theme 1 | West Bengal, India*

## Packages/Lib Installation & Imports

### Sloved an Major Import Error
*The first step in setting up the environment involves installing necessary Python packages. This is achieved using the !pip install command, which downloads and installs packages from PyPI (Python Package Index). The -t /kaggle/temp/custom_libs flag specifies that these packages should be installed into a temporary directory /kaggle/temp/custom_libs, rather than the default site-packages directory. This is often done in environments like Kaggle or Colab to manage dependencies or avoid conflicts.*

In [1]:
!pip install terratorch earthaccess -t /kaggle/temp/custom_libs -qqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import sys
sys.path.insert(0,"/kaggle/temp/custom_libs")

In [3]:
import os
import sys
import glob
import time
import zipfile
import warnings
import subprocess

import torch
import numpy as np
import pandas as pd
import albumentations as A
import lightning.pytorch as pl
import rasterio
import earthaccess
import terratorch.datamodules

from pathlib import Path
from albumentations.pytorch import ToTensorV2
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from terratorch.tasks import SemanticSegmentationTask
from kaggle_secrets import UserSecretsClient
from lightning.pytorch.callbacks import LearningRateMonitor

# from litlogger import LightningLogger
warnings.filterwarnings('ignore')
import cv2
from torch.utils.data import Dataset

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# Configuration Class
*All hyperparameters, file paths, and dataset specifications are centralised in a single Config class. This design ensures every component of the pipeline draws from one consistent source of truth, eliminating configuration drift between training, evaluation, and inference stages.*

In [4]:
# ==========================================
# 1. CONFIGURATION (Single Source of Truth)
# ==========================================
class Config:
    # Paths
    #######################
    # Put data into Temp!
    ########################
    DATA_ROOT = "./data"
    OUTPUT_DIR = "./Output"
    CHECKPOINT_DIR = "./Output/checkpoints"
    PRED_INPUT = "./data/prediction/image_10band"
    PRED_OUTPUT = "./prediction"
    SUBMISSION_CSV = "./submission.csv"

    # Dataset Specs
    NUM_CLASSES = 3
    # Means/Stds for HH, HV, Green, Red, NIR, SWIR
    # how to verify if 0 is a band and 7 is not ! 
    BANDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    IMAGE_PATTERN = "*image.tif"
    LABEL_PATTERN = "*label.tif"

    MEANS = [801.7887884673161, 357.10271382049245, 24.368360499649025, 2.307618329845533, 2.3790761222558037, 0.5562953557280781, 1836.6329517301992, 1688.3821636214086, 1812.1369410164718, 1275.3098927222795]
    STDS  = [435.3025284270273, 164.0301948857962, 9.40578525900619, 0.894201125347221, 2.3787784998722246, 0.4419584239191811, 628.8164877359038, 615.6047361975952, 588.7599355178615, 543.9400197036559]

    # Hardware / DataLoader
    BATCH_SIZE = 6
    NUM_WORKERS = 0
    PIN_MEMORY = True
    PERSISTENT_WORKERS = False

    # Hyperparameters
    LR = 3e-6
    WEIGHT_DECAY = 0.01
    MAX_EPOCHS = 120
    ACCUMULATE_GRAD = 4
    PRECISION = "16-mixed"
    SEED = 42

    # Architecture
    BACKBONE = "prithvi_eo_v2_100_tl"
    DECODER = "UperNetDecoder"
    DECODER_CHANNELS = 512
    DROPOUT = 0.4
    FREEZE_BACKBONE = False

    CLASS_WEIGHTS = [1.0, 7.0, 4.0]

# Data Pipeline Functions

### Data Sources and External Inputs

`NASADEM Digital Elevation Model (Band 7)`

*Along with the provided 6 bands, we integrated a 7th band representing elevation. This elevation data is fetched using NASA’s earthaccess library, which provides easy and free access to elevation data for the West Bengal region.*

*NASADEM provides 30 m resolution elevation data derived from NASA’s Shuttle Radar Topography Mission (SRTM). The data is accessed programmatically through the earthaccess NASA Earthdata client. Tiles covering the West Bengal bounding box (86.5–89.0°E, 21.5–24.5°N) are downloaded, mosaicked, and co-registered with the Sentinel imagery coordinate system.*

*The exact coordinates for downloading the elevation tiles are extracted from the metadata of the existing satellite images to ensure spatial alignment.*

In [5]:
def download_kaggle_competition(comp_name, delete_zip=True):
    """
    Download a Kaggle competition dataset.
    Cleans up after download if delete_zip is True.
    """
    print("Note: If you are trying to replicate this project, you must use your own data, as the data provided was privately owned by IBM.")
    zip_file = f"{comp_name}.zip"

    subprocess.run(
        ["kaggle", "competitions", "download", "-c", comp_name],
        check=True
    )

    time.sleep(10)

    if os.path.exists(zip_file):
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(".")

    if delete_zip and os.path.exists(zip_file):
        os.remove(zip_file)

    print(f"{comp_name} downloaded and prepared.")

In [6]:
class DEMDownloader:
    def __init__(self, download_dir):
        self.download_dir = download_dir
        os.makedirs(self.download_dir, exist_ok=True)
        self.auth = earthaccess.login(strategy="environment", persist=True)

    def download_wb_tiles(self):
        # This BBOX covers the training and prediction area for West Bengal
        WB_BBOX = (86.5, 21.5, 89.0, 24.5)

        search_results = earthaccess.search_data(
            short_name='NASADEM_HGT',
            bounding_box=WB_BBOX
        )

        if not search_results:
            print("No tiles found!")
            return []

        print(f"Found {len(search_results)} tiles. Starting download to {self.download_dir}...")
        downloaded_files = earthaccess.download(search_results, self.download_dir)
        return downloaded_files

In [7]:
class DEMProcessor:
    def __init__(self, raw_dir, temp_dir, aligned_dir):
        self.raw_dir = raw_dir
        self.temp_dir = temp_dir
        self.aligned_dir = aligned_dir
        for d in [temp_dir, aligned_dir]: os.makedirs(d, exist_ok=True)

    def extract_zips(self):
        print("📦 Extracting DEM packages...")
        for z in glob.glob(os.path.join(self.raw_dir, "*.zip")):
            with zipfile.ZipFile(z, 'r') as ref:
                ref.extractall(self.temp_dir)

    def get_mosaic(self):
        hgt_files = glob.glob(os.path.join(self.temp_dir, "**/*.hgt"), recursive=True)
        src_files = [rasterio.open(f) for f in hgt_files]
        mosaic, transform = merge(src_files, method="first")
        return mosaic, transform, src_files[0].crs, src_files

In [8]:
class DeepFeatureStacker:
    def __init__(self, output_dir, mosaic_data, mosaic_trans, mosaic_crs, target_crs):
        self.output_dir = output_dir
        from rasterio.warp import calculate_default_transform
        
        dst_trans, dst_width, dst_height = calculate_default_transform(
            mosaic_crs, target_crs, mosaic_data.shape[2], mosaic_data.shape[1],
            *rasterio.transform.array_bounds(mosaic_data.shape[1], mosaic_data.shape[2], mosaic_trans)
        )
        self.cached_dem = np.zeros((1, dst_height, dst_width), dtype='float32')
        reproject(
            source=mosaic_data, destination=self.cached_dem,
            src_transform=mosaic_trans, src_crs=mosaic_crs,
            dst_transform=dst_trans, dst_crs=target_crs,
            resampling=Resampling.bilinear
        )
        self.cached_trans = dst_trans

    def calculate_slope(self, dem):
        px, py = np.gradient(dem)
        slope = np.sqrt(px**2 + py**2)
        return slope

    def create_10band_stack(self, img_path):
        file_id = os.path.basename(img_path).replace("_image.tif", "")
        out_path = os.path.join(self.output_dir, f"{file_id}_image.tif")

        with rasterio.open(img_path) as ref:
            img_data = ref.read().astype(np.float32)
            meta = ref.meta.copy()
            meta.update(count=10, dtype='float32')

            from rasterio.windows import from_bounds
            window = from_bounds(*ref.bounds, transform=self.cached_trans)
            r, c = int(window.row_off), int(window.col_off)
            h, w = int(window.height), int(window.width)
            
            dem_crop = self.cached_dem[0, r:r+h, c:c+w]

            # ✅ Safety Check for edge patches
            if dem_crop.size == 0 or dem_crop.shape[0] == 0:
                dem_data = np.zeros((ref.height, ref.width), dtype='float32')
                slope_data = np.zeros((ref.height, ref.width), dtype='float32')
            else:
                dem_data = cv2.resize(dem_crop, (ref.width, ref.height))
                slope_data = self.calculate_slope(dem_data)

            # SAR Math (DeepSARFlood Logic)
            hh, hv = img_data[0], img_data[1]
            diff_band = np.clip(hh - hv, 1e-6, None)
            new_band_1 = 10 * np.log10(diff_band)
            new_band_2 = hh / (hv + 1e-6)

            # Assemble: HH, HV, Diff, Ratio, DEM, Slope, G, R, NIR, SWIR
            final_stack = np.stack([
                hh, hv, new_band_1, new_band_2, 
                dem_data, slope_data,
                img_data[2], img_data[3], img_data[4], img_data[5]
            ])

            with rasterio.open(out_path, 'w', **meta) as dst:
                dst.write(final_stack)
        return out_path

In [9]:
def run_data_pipeline():
    """Consolidated 10-band pipeline for Train and Prediction."""
    download_kaggle_competition("anrfaisehack-theme-1-phase2")
    
    downloader = DEMDownloader(f"{Config.DATA_ROOT}/dem_raw")
    downloader.download_wb_tiles()
    dp = DEMProcessor(f"{Config.DATA_ROOT}/dem_raw", f"{Config.DATA_ROOT}/dem_temp", f"{Config.DATA_ROOT}/dem_aligned")
    dp.extract_zips()
    mosaic, trans, crs, opened_files = dp.get_mosaic()

    train_images = glob.glob(f"{Config.DATA_ROOT}/image/*.tif")
    pred_images = glob.glob(f"{Config.DATA_ROOT}/prediction/image/*.tif")
    
    with rasterio.open(train_images[0]) as sample:
        target_crs = sample.crs

    # Single stacker instance
    core_stacker = DeepFeatureStacker(
        output_dir=f"{Config.DATA_ROOT}/image_10band", 
        mosaic_data=mosaic, mosaic_trans=trans, mosaic_crs=crs, target_crs=target_crs
    )

    print(f"🚀 Stacking {len(train_images)} training images...")
    os.makedirs(f"{Config.DATA_ROOT}/image_10band", exist_ok=True)
    for img in train_images:
        core_stacker.create_10band_stack(img)

    print(f"🚀 Stacking {len(pred_images)} prediction images...")
    os.makedirs(Config.PRED_INPUT, exist_ok=True)
    core_stacker.output_dir = Config.PRED_INPUT
    for img in pred_images:
        core_stacker.create_10band_stack(img)

    for f in opened_files: f.close()
    print("✅ 10-Band Optimized Pipeline Success!")

# Training Pipeline Functions

In [10]:
# ==========================================
# 3. UTILITIES: RLE & SUBMISSION
# ==========================================
# Convert prediction tif files to Kaggle-style run-length encoding (RLE) Submission csv
def mask_to_rle(mask):
    """
    Convert binary mask to RLE (Kaggle format).
    Mask must be 2D numpy array with values 0 or 1.
    """
    pixels = mask.flatten(order="F")  # column-major
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

def generate_submission(tif_dir, output_csv):
    """
    Read TIF predictions and generate Kaggle submission CSV.
    Logic kept identical to competition standards.
    """
    tif_dir = Path(tif_dir)
    rows = []

    for tif_path in sorted(tif_dir.glob("*.tif")):
        with rasterio.open(tif_path) as src:
            mask = src.read(1)

        # UPDATE THIS LINE: Only target the specific class the competition wants
        # Change the '1' to a '2' if class 2 is the actual flood class!
        TARGET_CLASS = 1 
        mask = (mask == TARGET_CLASS).astype(np.uint8)

        rle = mask_to_rle(mask)

        rows.append({
            "id": tif_path.name.replace("_image.tif", ""),
            "rle_mask": rle
        })

    df = pd.DataFrame(rows)
    df = df.replace("", 0).fillna(0) # replace null/ na with zero - kaggle compatible
    df.to_csv(output_csv, index=False)
    print(f"Saved Kaggle RLE CSV : {output_csv}")

In [11]:
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.CoarseDropout(
            num_holes_range=(1,4),
            hole_height_range=(8,32),
            hole_width_range=(8,32),
            fill=0,
            p=0.2
        ),

        ToTensorV2(),

    ])

### Model Architecture

`Decoder — UperNet (512 Channels)`

*A **UperNet decoder** is used to reconstruct the segmentation map.
It combines multi-scale encoder features*

---

`Loss Function`

*A **hybrid loss** is used to improve segmentation quality:*

| Loss Component         | Weight | Purpose                                              |
| ---------------------- | ------ | ---------------------------------------------------- |
| **Dice Loss**          | 0.6    | Handles class imbalance and optimizes region overlap |
| **Cross-Entropy Loss** | 0.4    | Provides stable per-pixel training                   |

*This combination ensures **stable optimization while improving flood boundary delineation.*

---

`Optimizer & Learning Strategy`

* **Optimizer:** AdamW
* **Learning Rate:** 2 × 10⁻⁵
* **Weight Decay:** 0.01
* **Scheduler:** CosineAnnealingWarmRestarts

*The **low learning rate preserves the pre-trained geospatial representations** while allowing effective domain adaptation during fine-tuning.*

In [12]:
# ==========================================
# 4. DATAMODULE & MODEL BUILDERS
# ==========================================
def build_datamodule(cfg):
    return terratorch.datamodules.GenericNonGeoSegmentationDataModule(
        batch_size=cfg.BATCH_SIZE,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        persistent_workers=cfg.PERSISTENT_WORKERS,
        num_classes=cfg.NUM_CLASSES,
        train_data_root=f"{cfg.DATA_ROOT}/image_10band",
        train_label_data_root=f"{cfg.DATA_ROOT}/label",
        val_data_root=f"{cfg.DATA_ROOT}/image_10band",
        val_label_data_root=f"{cfg.DATA_ROOT}/label",
        test_data_root=f"{cfg.DATA_ROOT}/image_10band",
        test_label_data_root=f"{cfg.DATA_ROOT}/label",
        train_split=f"{cfg.DATA_ROOT}/split/train.txt",
        val_split=f"{cfg.DATA_ROOT}/split/val.txt",
        test_split=f"{cfg.DATA_ROOT}/split/test.txt",
        img_grep=cfg.IMAGE_PATTERN,
        label_grep=cfg.LABEL_PATTERN,
        train_transform=get_train_transforms(),
        means=cfg.MEANS,
        stds=cfg.STDS,
        no_data_replace=0,
        no_label_replace=-1,
        predict_data_root=cfg.PRED_INPUT
    )

def build_model(cfg):
    model_args = {
        "backbone": cfg.BACKBONE,
        "backbone_pretrained": True,
        "backbone_bands": cfg.BANDS,
        "backbone_num_frames": 1,
        "decoder": cfg.DECODER,
        "decoder_channels": cfg.DECODER_CHANNELS,
        "decoder_scale_modules": True,
        "num_classes": cfg.NUM_CLASSES,
        "head_dropout": cfg.DROPOUT,
        "rescale": True,
        # Match the helper code's simplified neck for V2 models
        "necks": [
        dict(
            name="ReshapeTokensToImage",
            effective_time_dim=1,
        )
    ]
}

    return SemanticSegmentationTask(
        model_args=model_args,
        plot_on_val=False,
        class_weights=cfg.CLASS_WEIGHTS,
        loss = {"ce": 0.4, "dice": 0.6},
        lr=cfg.LR,
        optimizer="AdamW",
        optimizer_hparams={"weight_decay": cfg.WEIGHT_DECAY},
        ignore_index=-1,
        freeze_backbone=cfg.FREEZE_BACKBONE,
        model_factory="EncoderDecoderFactory",
        scheduler="ReduceLROnPlateau",
        # scheduler_hparams={"T_mult": 2},
    )

def get_loggers(cfg):
    loggers = [
        TensorBoardLogger(save_dir=cfg.OUTPUT_DIR, name="tb_logs"),
        CSVLogger(save_dir=cfg.OUTPUT_DIR, name="csv_logs")
    ]
    return loggers


`Transformer Overview`

![Training Pipeline Overview](https://i.ibb.co/V0Y4Ds42/image.png)

In [13]:
import torch.nn.functional as F

# ==========================================
# 6. SUBMISSION GENERATOR (Phase 2 Optimized)
# ==========================================
def generate_phase2_submission(tif_dir, output_csv):
    """
    Specifically targets Class 1 (Flood) for RLE.
    Handles '0 0' requirement for empty masks.
    """
    tif_dir = Path(tif_dir)
    rows = []

    for tif_path in sorted(tif_dir.glob("*.tif")):
        with rasterio.open(tif_path) as src:
            mask = src.read(1)

        # PHASE 2 CRITICAL: Target ONLY Class 1 (Flood)
        # 0: No Flood, 1: Flood, 2: Water Body
        binary_flood_mask = (mask == 1).astype(np.uint8)

        # mask_to_rle should be defined globally as per your previous cells
        rle = mask_to_rle(binary_flood_mask)
        
        # HACKATHON RULE: If empty, must be "0 0"
        if not rle or rle.strip() == "":
            rle = "0 0"

        rows.append({
            "id": tif_path.name.replace("_image.tif", ""),
            "rle_mask": rle
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"🚀 FINAL SUBMISSION READY: {output_csv}")

# ==========================================
# 5. MAIN EXECUTION PIPELINE
# ==========================================
def main():

    cfg = Config()
    pl.seed_everything(cfg.SEED)
    for d in [cfg.CHECKPOINT_DIR, cfg.PRED_OUTPUT]: os.makedirs(d, exist_ok=True)
    torch.set_float32_matmul_precision("high")

    # 3. Initialize Model and Trainer
    dm = build_datamodule(cfg)
    model = build_model(cfg)

    checkpoint_cb = ModelCheckpoint(
        monitor="val/IoU_1",   # ✅ FIXED (correct key)
        mode="max", 
        dirpath=cfg.CHECKPOINT_DIR,
        filename="best-flood-{epoch:02d}-{val/IoU_1:.4f}",  # ✅ also fix here
        save_top_k=1,
    )
    
    trainer = pl.Trainer(
        accelerator="gpu", devices=1, precision=cfg.PRECISION,
        max_epochs=cfg.MAX_EPOCHS, accumulate_grad_batches=cfg.ACCUMULATE_GRAD,
        logger=get_loggers(cfg), 
        callbacks=[checkpoint_cb, LearningRateMonitor(logging_interval='step')],
        num_sanity_val_steps=2, log_every_n_steps=5
    )

    # 4. Training Phase
    print(">>> Starting Training...")
    trainer.fit(model, datamodule=dm)

    # 5. Validation/Test with Best Checkpoint
    best_path = checkpoint_cb.best_model_path
    if best_path:
        print(f">>> Testing with best checkpoint: {best_path}")
        trainer.test(model, datamodule=dm, ckpt_path=best_path)

    # 6. Prediction & Uncertainty Logic (DeepSARFlood Approach)
    print(">>> Generating Predictions and Confidence Maps...")
    model.eval()
    dm.setup("predict")

    UNCERTAINTY_DIR = os.path.join(cfg.OUTPUT_DIR, "uncertainty_maps")
    os.makedirs(UNCERTAINTY_DIR, exist_ok=True)

    # Note: Use best_path to ensure we are predicting with the top-performing model
    predictions = trainer.predict(model, datamodule=dm, ckpt_path=best_path if best_path else None)

    for batch_idx, (logits, file_paths) in enumerate(predictions):
        if isinstance(logits, tuple): logits = logits[0]
        
        # Softmax to get probabilities for all 3 classes
        preds_class = logits.cpu().numpy().astype("int16")
        
        uncertainty = np.zeros_like(preds_class, dtype="float32")

        for i in range(preds_class.shape[0]):
            ref_path = file_paths[i]
            base_name = os.path.basename(ref_path)
            arr = preds_class[i]
            unc_arr = uncertainty[i]

            # Handle no-data values correctly
            arr[arr < 0] = -1
            
            with rasterio.open(ref_path) as src:
                meta = src.meta.copy()

            # --- Save the 3-Class Segmentation Result ---
            meta.update({"count": 1, "dtype": "int16", "nodata": -1, "compress": "lzw"})
            out_path = os.path.join(cfg.PRED_OUTPUT, base_name)
            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(arr, 1)

            # --- Save the Uncertainty Map ---
            meta.update({"dtype": "float32", "nodata": -1})
            unc_out_path = os.path.join(UNCERTAINTY_DIR, base_name.replace(".tif", "_uncertainty.tif"))
            with rasterio.open(unc_out_path, "w", **meta) as dst:
                dst.write(unc_arr, 1)

    # 7. Final Submission CSV
    generate_phase2_submission(cfg.PRED_OUTPUT, cfg.SUBMISSION_CSV)
    print(">>> Pipeline Finished Successfully!")

In [14]:
import os
import urllib.request
import time

# 1. Start TensorBoard in the background
print("🚀 Starting TensorBoard in the background...")
os.system("tensorboard --logdir ./Output --host 0.0.0.0 --port 6005 &")

# 2. Install localtunnel silently
print("📦 Installing localtunnel...")
os.system("npm install -g localtunnel -qqq")

# 3. Fetch the Kaggle machine's external IP
endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print("="*50)
print(f"🛑 IMPORTANT: Copy this IP address: {endpoint_ip}")
print("You will need to paste this as the 'Endpoint IP' on the warning page.")
print("="*50)

# 4. Start the reverse tunnel IN THE BACKGROUND
# Using 'nohup' and '&' completely detaches it from the cell execution
print("🔗 Creating tunnel in the background...")
os.system("nohup lt --port 6005 > tunnel_url.txt 2>&1 &")

# Give the tunnel 4 seconds to connect to the server and generate the link
time.sleep(4)

# 5. Read the generated URL from the text file and print it
print("✅ Tunnel is running! Here is your link:")
with open("tunnel_url.txt", "r") as f:
    print(f.read().strip())

print("\n🎉 Done! This cell is now free and you can run other cells.")

🚀 Starting TensorBoard in the background...
📦 Installing localtunnel...


/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:292: SyntaxWarning: invalid escape sequence '\s'
  "[`\000-\040\177-\240\s]+",
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:339: SyntaxWarning: invalid escape sequence '\s'
  style = re.compile('url\s*\(\s*[^\s)]+?\s*\)\s*').sub(' ', style)
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:354: SyntaxWarning: invalid escape sequence '\s'
  if not re.match("^\s*([-\w]+\s*:[^:;]*(;\s*|$))*$", style):
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:358: SyntaxWarning: invalid escape sequence '\w'
  for prop, value in re.findall('([-\w]+)\s*:\s*([^:;]*)', style):



added 22 packages in 2s

3 packages are looking for funding
  run `npm fund` for details
🛑 IMPORTANT: Copy this IP address: 136.115.41.86
You will need to paste this as the 'Endpoint IP' on the warning page.
🔗 Creating tunnel in the background...


npm notice
npm notice New major version of npm available! 10.8.2 -> 11.12.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.12.1
npm notice To update run: npm install -g npm@11.12.1
npm notice
2026-04-02 14:38:39.956641: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775140720.183156     199 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775140720.247043     199 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775140720.760412     199 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775140720.760458     199 computation_p

✅ Tunnel is running! Here is your link:
your url is: https://dry-results-learn.loca.lt

🎉 Done! This cell is now free and you can run other cells.


In [15]:
if __name__ == "__main__":

    # 1. Environment Setup
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"

    # ── Credentials ───────────────────────────────────────────────────────
    # NEVER hardcode secrets in a notebook — it ships to git and to Kaggle.
    #   EARTHDATA_USERNAME / EARTHDATA_PASSWORD : NASA Earthdata (NASADEM tiles)
    #                                             free signup at urs.earthdata.nasa.gov
    #   LIGHTNING_API_KEY                       : optional, Lightning logging
    #
    # On Kaggle : Add-ons ▸ Secrets, using these exact names.
    # Locally   : export them as environment variables before launching Jupyter.
    REQUIRED = ("EARTHDATA_USERNAME", "EARTHDATA_PASSWORD")
    OPTIONAL = ("LIGHTNING_API_KEY",)

    try:
        _secrets = UserSecretsClient()
        for _key in REQUIRED + OPTIONAL:
            try:
                os.environ[_key] = _secrets.get_secret(_key)
            except Exception:
                pass  # not set as a Kaggle Secret; fall back to the environment
    except Exception:
        pass  # not running on Kaggle — rely on environment variables alone

    _missing = [k for k in REQUIRED if not os.environ.get(k)]
    if _missing:
        raise RuntimeError(
            f"Missing credential(s): {', '.join(_missing)}. "
            "Register free at https://urs.earthdata.nasa.gov/ and provide them "
            "as Kaggle Secrets or environment variables."
        )

    # 2. Data Preparation
    run_data_pipeline()

    import gc
    import cv2
    cv2.setNumThreads(0)
    gc.collect()
    torch.cuda.empty_cache()
    main()

Note: If you are trying to replicate this project, you must use your own data, as the data provided was privately owned by IBM.


  1%|          | 7.00M/626M [00:00<00:09, 71.6MB/s]

100%|██████████| 626M/626M [00:03<00:00, 200MB/s]



NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

TensorBoard 2.19.0 at http://0.0.0.0:6005/ (Press CTRL+C to quit)


anrfaisehack-theme-1-phase2 downloaded and prepared.


...

Restoring states from the checkpoint path at /kaggle/working/Output/checkpoints/best-flood-epoch=12-val/IoU_1=0.1635.ckpt


>>> Generating Predictions and Confidence Maps...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/Output/checkpoints/best-flood-epoch=12-val/IoU_1=0.1635.ckpt
2026-04-02 14:57:16,173 - INFO - Checking stackability for predict split.


Output()

🚀 FINAL SUBMISSION READY: ./submission.csv
>>> Pipeline Finished Successfully!
